# feature analysis

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. This notebook reads the corrected selection and existing feature caches; it does not require newly trained NPE checkpoints.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/feature_analysis`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'feature_analysis')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

The goal of this notebook is to check different relations within the simulations. And check importance of different features on the target m200c using permutation importance and random forest. This is done on a set of features with core only: vr, vt, seperation and log(Mi/Mj) where Mi and Mj are the stellar masses. core features + chemical features: metallicity mean and metallicity difference. core features + morphology: sigma_mean, sigma_difference and kappa_mean and kappa_difference where sigma is the and kappa is the . core features + photometry: gr_mean and gr_diff.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pickle
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import r2_score

In [ ]:
# Loading FLAMINGO and some TNG300
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

In [ ]:
# Loading TNG300
tng_dir = PROJECT / "analogues/tng_for_comparison_v3"

keep_idx    = TNG_DATA["keep_idx"]
tng_pairs_i = TNG_DATA["pairs_i"]
tng_pairs_j = TNG_DATA["pairs_j"]

# galaxy-level — use particle-computed fields only (FLAMINGO-equivalent definitions)
tng_gr_sample   = TNG_DATA["g_r"]
tng_met_sample  = TNG_DATA["metallicity"]     # particle-level, no aperture
tng_mass_sample = TNG_DATA["stellar_mass"]

# particle-computed fields (from pkl, keyed by global subhalo ID)
tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]

tng_sigma_i = finite_lookup(tng_sigma_star, tng_pairs_i_global, "tng_sigma_star")
tng_sigma_j = finite_lookup(tng_sigma_star, tng_pairs_j_global, "tng_sigma_star")
tng_kappa_i = finite_lookup(kappa_cache, tng_pairs_i_global, "kappa_cache")
tng_kappa_j = finite_lookup(kappa_cache, tng_pairs_j_global, "kappa_cache")

# pair-level
tng_separation   = TNG_DATA["separation"]
tng_vr           = TNG_DATA["vr"]
tng_vt           = TNG_DATA["vt"]
tng_log_m200c    = TNG_DATA["log_m200c"]
tng_log_mass_ratio = TNG_DATA["log_mass_ratio"]

# NOTE: don't use dir/"vel_disp.npy" that's the catalog all-particle field, 
# not equivalent to FLAMINGO. tng_sigma_star pkl above instead is equivalent to FLAMINGO.

In [ ]:
# For symmetry let's represent each pair as (mean, |diff|) of galaxy-level features
# so pair (i,j) and (j,i) will give identical feature vectors

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]

    gr_i    = sample["g_r"][pi];          gr_j    = sample["g_r"][pj]
    sigma_i = sample["vel_disp"][pi];     sigma_j = sample["vel_disp"][pj]
    kap_i   = sample["kappa_rot"][pi];    kap_j   = sample["kappa_rot"][pj]
    met_i   = met_transform["flamingo_met_dex"][pi]
    met_j   = met_transform["flamingo_met_dex"][pj]

    df = pd.DataFrame({
        "gr_mean":        (gr_i    + gr_j)    / 2,
        "gr_diff":        np.abs(gr_i    - gr_j),
        "met_mean":       (met_i   + met_j)   / 2,
        "met_diff":       np.abs(met_i   - met_j),
        "sigma_mean":     (sigma_i + sigma_j) / 2,
        "sigma_diff":     np.abs(sigma_i - sigma_j),
        "kappa_mean":     (kap_i   + kap_j)   / 2,
        "kappa_diff":     np.abs(kap_i   - kap_j),
        "separation":     pairs["separation_kpc"],
        "vr":             pairs["vr"],
        "vt":             pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"],
        "log_m200c":      pairs["log_tot_virial_mass"],
    })
    return df

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):

    gr_i  = tng_gr_sample[tng_pairs_i];  gr_j  = tng_gr_sample[tng_pairs_j]

    # Common physical dex convention, matching the inference feature definition.
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]

    df = pd.DataFrame({
        "gr_mean":    (gr_i        + gr_j)        / 2,
        "gr_diff":    np.abs(gr_i        - gr_j),
        "met_mean":   (met_i       + met_j)        / 2,
        "met_diff":   np.abs(met_i       - met_j),
        "sigma_mean": (tng_sigma_i + tng_sigma_j)  / 2,
        "sigma_diff": np.abs(tng_sigma_i - tng_sigma_j),
        "kappa_mean": (tng_kappa_i + tng_kappa_j)  / 2,
        "kappa_diff": np.abs(tng_kappa_i - tng_kappa_j),
        "separation": tng_separation,
        "vr":         tng_vr,
        "vt":         tng_vt,
        "log_mass_ratio": tng_log_mass_ratio,
        "log_m200c":  tng_log_m200c,               # target for now will add some more later
    })
    return df

flam_df = pair_features_flamingo(sample, pairs, met_transform)
tng_df  = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)

flam_df["host_i"] = sample["host_top_index"][pairs["i"]]
flam_df["host_j"] = sample["host_top_index"][pairs["j"]]
tng_df["host_i"] = TNG_DATA["grnr"][tng_pairs_i]
tng_df["host_j"] = TNG_DATA["grnr"][tng_pairs_j]

FEATURES = ["gr_mean", "gr_diff", "met_mean", "met_diff",
            "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
            "separation", "vr", "vt", "log_mass_ratio"]
TARGET = "log_m200c"

# dropping rows with any null values which they arent
flam_df_clean = flam_df.dropna(subset=FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=FEATURES + [TARGET])
print(f"FLAMINGO pairs (clean): {len(flam_df_clean):,}")
print(f"TNG     pairs (clean):  {len(tng_df_clean):,}")
assert len(tng_df_clean) == TNG_MANIFEST["pair_count"], "Unexpected loss of corrected TNG rows."
flam_groups = pair_groups(flam_df_clean)
tng_groups = pair_groups(tng_df_clean)

In [ ]:
# Pairwise relation plots 
fig, axes = plt.subplots(6, 2, figsize=(14, 36))
fig.suptitle("Pairwise Relations — FLAMINGO vs TNG300", fontsize=14)

plot_pairs = [
    ("vt",        "vr",         "$v_t$ [km/s]",              "$v_r$ [km/s]"),
    ("separation","vr",         "Separation [kpc]",           "$v_r$ [km/s]"),
    ("separation","vt",         "Separation [kpc]",           "$v_t$ [km/s]"),
    ("kappa_mean","sigma_mean", r"$\kappa_{\rm co}$ (mean)",  r"$\sigma_\star$ [km/s] (mean)"),
    ("met_mean",  "gr_mean",    "Metallicity scaled (mean)",  "g-r (mean)"),
    ("separation","log_m200c",  "Separation [kpc]",           r"$\log M_{200c}$"),
]

for row, (xkey, ykey, xlabel, ylabel) in enumerate(plot_pairs):
    for col, (df, color, label) in enumerate([
        (flam_df_clean, "#E07B3F", "FLAMINGO"),
        (tng_df_clean,  "#4C72B0", "TNG300-1"),
    ]):
        ax = axes[row, col]
        valid = df[xkey].notna() & df[ykey].notna()
        r, _ = spearmanr(df.loc[valid, xkey], df.loc[valid, ykey])
        hb = ax.hexbin(df.loc[valid, xkey], df.loc[valid, ykey],
                       gridsize=35, cmap="Greys", mincnt=1,
                       norm=mcolors.LogNorm(), linewidths=0.2)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f"{label}  Spearman r={r:.3f}", fontsize=10)
        plt.colorbar(hb, ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_07.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Spearman correlation matrix
all_cols = FEATURES + [TARGET]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Spearman Correlation Matrix — FLAMINGO vs TNG300", fontsize=13)

for ax, df, label in [
    (axes[0], flam_df_clean, "FLAMINGO"),
    (axes[1], tng_df_clean,  "TNG300-1"),
]:
    corr = df[all_cols].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle only

    sns.heatmap(
        corr,
        ax=ax,
        mask=mask,
        annot=True, fmt=".2f",
        cmap="RdBu_r", vmin=-1, vmax=1, center=0,
        linewidths=0.5,
        annot_kws={"size": 7},
        cbar_kws={"shrink": 0.8},
        square=True,
    )
    ax.set_title(label, fontsize=12)
    ax.tick_params(labelsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_08.pdf", bbox_inches="tight")
plt.show()

# flag features with large cross-sim correlation differences 
flam_corr_target = flam_df_clean[all_cols].corr(method="spearman")[TARGET].drop(TARGET)
tng_corr_target  = tng_df_clean[all_cols].corr(method="spearman")[TARGET].drop(TARGET)

diff = (tng_corr_target - flam_corr_target).abs().sort_values(ascending=False)

print("Feature correlation with log_m200c:")
print(f"{'Feature':<16} {'FLAMINGO':>10} {'TNG':>10} {'|Δr|':>8}")
print("-" * 48)
for feat in diff.index:
    print(f"{feat:<16} {flam_corr_target[feat]:>10.3f} "
          f"{tng_corr_target[feat]:>10.3f} {diff[feat]:>8.3f}")

In [ ]:
# Project-specific joblib scratch space, portable to Roihu.
joblib_temp = RESULTS_DIR / "joblib_tmp"
joblib_temp.mkdir(parents=True, exist_ok=True)
os.environ["JOBLIB_TEMP_FOLDER"] = str(joblib_temp)

In [ ]:
# RF just for testing we will train on FLAMINGO, CV + test on TNG 
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import r2_score
import warnings

RF_PARAMS = dict(n_estimators=500, max_features="sqrt",
                 random_state=42, n_jobs=-1)

X_flam = flam_df_clean[FEATURES].values
y_flam = flam_df_clean[TARGET].values

X_tng  = tng_df_clean[FEATURES].values
y_tng  = tng_df_clean[TARGET].values

# 5-fold CV on FLAMINGO 
kf = GroupKFold(n_splits=5)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_flam, groups=flam_groups)):
    rf = RandomForestRegressor(**RF_PARAMS)
    rf.fit(X_flam[train_idx], y_flam[train_idx])
    r2 = r2_score(y_flam[val_idx], rf.predict(X_flam[val_idx]))
    cv_scores.append(r2)
    print(f"  Fold {fold+1}: R² = {r2:.4f}")

print(f"\nFLAMINGO 5-fold CV:  R² = {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# Fit on full FLAMINGO, test on TNG
rf_full = RandomForestRegressor(**RF_PARAMS)
rf_full.fit(X_flam, y_flam)

y_tng_pred = rf_full.predict(X_tng)
r2_tng = r2_score(y_tng, y_tng_pred)
print(f"TNG300 test (FLAMINGO-trained RF): R² = {r2_tng:.4f}")
print(f"Cross-sim R² drop: {np.mean(cv_scores) - r2_tng:.4f}")

# Predicted vs actual plot 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("RF Predictions vs True $\\log M_{200c}$", fontsize=13)

# FLAMINGO (last fold for illustration)
rf_plot = RandomForestRegressor(**RF_PARAMS)
train_idx, val_idx = group_split(flam_df_clean, test_size=0.2, seed=42)
X_tr, X_val = X_flam[train_idx], X_flam[val_idx]
y_tr, y_val = y_flam[train_idx], y_flam[val_idx]
rf_plot.fit(X_tr, y_tr)
y_val_pred = rf_plot.predict(X_val)

for ax, y_true, y_pred, color, label, r2 in [
    (axes[0], y_val,  y_val_pred, "#E07B3F", "FLAMINGO (20% holdout)", r2_score(y_val, y_val_pred)),
    (axes[1], y_tng,  y_tng_pred, "#4C72B0", "TNG300-1 (full, FLAMINGO-trained)", r2_tng),
]:
    lims = [min(y_true.min(), y_pred.min()) - 0.1,
            max(y_true.max(), y_pred.max()) + 0.1]
    ax.hexbin(y_true, y_pred, gridsize=35, cmap="Greys",
              mincnt=1, norm=mcolors.LogNorm(), linewidths=0.2)
    ax.plot(lims, lims, "r--", lw=1.5, label="1:1")
    ax.set_xlabel(r"True $\log M_{200c}$", fontsize=11)
    ax.set_ylabel(r"Predicted $\log M_{200c}$", fontsize=11)
    ax.set_title(f"{label}\n$R^2={r2:.4f}$", fontsize=11)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=9)
    plt.colorbar(ax.collections[0], ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_10_1.pdf", bbox_inches="tight")
plt.show()

# ── 4. Permutation importance — FLAMINGO holdout vs TNG ───────────────────
perm_flam = permutation_importance(
    rf_plot, X_val, y_val,
    n_repeats=30, random_state=42, n_jobs=1
)
perm_tng = permutation_importance(
    rf_full, X_tng, y_tng,
    n_repeats=30, random_state=42, n_jobs=1
)

# sort by FLAMINGO importance
sort_idx = perm_flam.importances_mean.argsort()[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Permutation Importance — FLAMINGO holdout vs TNG300 test", fontsize=13)

for ax, perm, color, label in [
    (axes[0], perm_flam, "#E07B3F", "FLAMINGO (holdout)"),
    (axes[1], perm_tng,  "#4C72B0", "TNG300-1 (test)"),
]:
    means = perm.importances_mean[sort_idx]
    stds  = perm.importances_std[sort_idx]
    feats = [FEATURES[i] for i in sort_idx]
    ax.barh(feats[::-1], means[::-1], xerr=stds[::-1],
            color=color, alpha=0.8, capsize=4)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Mean R² decrease", fontsize=11)
    ax.set_title(label, fontsize=11)
    ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_10_2.pdf", bbox_inches="tight")
plt.show()

# Summary table 
print(f"\n{'Feature':<16} {'PI_FLAM':>10} {'PI_TNG':>10} {'Δ':>10}  "
      f"{'Wass_norm':>10}  {'Flag'}")
print("-" * 70)

wass_dict = {feature: normalized_wasserstein(flam_df_clean[feature].values,
                                              tng_df_clean[feature].values)
             for feature in FEATURES}

for i in sort_idx:
    feat  = FEATURES[i]
    pi_f  = perm_flam.importances_mean[i]
    pi_t  = perm_tng.importances_mean[i]
    delta = pi_t - pi_f
    wass  = wass_dict.get(feat, np.nan)
    flag  = ("⚠ gap" if (not np.isnan(wass) and wass > 0.30 and pi_f > 0.01)
             else "ok")
    wass_str = f"{wass:.2f}" if not np.isnan(wass) else "  —"
    print(f"{feat:<16} {pi_f:>10.4f} {pi_t:>10.4f} {delta:>10.4f}  "
          f"{wass_str:>10}  {flag}")

When using all features FLAMINGO seems to overestimate the virial mass compared to TNG300.


In [ ]:
# RF train TNG, CV + test FLAMINGO

# Report sample counts from the current data; connected hosts stay in one fold.
X_tng_all  = tng_df_clean[FEATURES].values
y_tng_all  = tng_df_clean[TARGET].values

X_flam_all = flam_df_clean[FEATURES].values
y_flam_all = flam_df_clean[TARGET].values

# 5-fold CV on TNG
kf = GroupKFold(n_splits=5)
cv_scores_tng = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_tng_all, groups=tng_groups)):
    rf = RandomForestRegressor(**RF_PARAMS)
    rf.fit(X_tng_all[train_idx], y_tng_all[train_idx])
    r2 = r2_score(y_tng_all[val_idx], rf.predict(X_tng_all[val_idx]))
    cv_scores_tng.append(r2)
    print(f"  Fold {fold+1}: R² = {r2:.4f}")

print(f"\nTNG300 5-fold CV:  R² = {np.mean(cv_scores_tng):.4f} ± {np.std(cv_scores_tng):.4f}")

# Fit on full TNG, test on FLAMINGO
rf_tng_full = RandomForestRegressor(**RF_PARAMS)
rf_tng_full.fit(X_tng_all, y_tng_all)

y_flam_pred = rf_tng_full.predict(X_flam_all)
r2_flam     = r2_score(y_flam_all, y_flam_pred)
print(f"FLAMINGO test (TNG-trained RF): R² = {r2_flam:.4f}")
print(f"Cross-sim R² drop: {np.mean(cv_scores_tng) - r2_flam:.4f}")

# Predicted vs actual 
train_idx, val_idx = group_split(tng_df_clean, test_size=0.2, seed=42)
X_tr, X_val = X_tng_all[train_idx], X_tng_all[val_idx]
y_tr, y_val = y_tng_all[train_idx], y_tng_all[val_idx]
rf_tng_plot = RandomForestRegressor(**RF_PARAMS)
rf_tng_plot.fit(X_tr, y_tr)
y_val_pred_tng = rf_tng_plot.predict(X_val)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("RF Predictions vs True $\\log M_{200c}$ — TNG-trained", fontsize=13)

for ax, y_true, y_pred, color, label, r2 in [
    (axes[0], y_val,      y_val_pred_tng, "#4C72B0", "TNG300 (20% holdout)",          r2_score(y_val, y_val_pred_tng)),
    (axes[1], y_flam_all, y_flam_pred,    "#E07B3F", "FLAMINGO (full, TNG-trained)",   r2_flam),
]:
    lims = [min(y_true.min(), y_pred.min()) - 0.1,
            max(y_true.max(), y_pred.max()) + 0.1]
    ax.hexbin(y_true, y_pred, gridsize=35, cmap="Greys",
              mincnt=1, norm=mcolors.LogNorm(), linewidths=0.2)
    ax.plot(lims, lims, "r--", lw=1.5, label="1:1")
    ax.set_xlabel(r"True $\log M_{200c}$", fontsize=11)
    ax.set_ylabel(r"Predicted $\log M_{200c}$", fontsize=11)
    ax.set_title(f"{label}\n$R^2={r2:.4f}$", fontsize=11)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=9)
    plt.colorbar(ax.collections[0], ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_12_1.pdf", bbox_inches="tight")
plt.show()

# Permutation importance 
perm_tng_holdout = permutation_importance(
    rf_tng_plot, X_val, y_val,
    n_repeats=30, random_state=42, n_jobs=1
)
perm_flam_test = permutation_importance(
    rf_tng_full, X_flam_all, y_flam_all,
    n_repeats=30, random_state=42, n_jobs=1
)

sort_idx = perm_tng_holdout.importances_mean.argsort()[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Permutation Importance — TNG holdout vs FLAMINGO test", fontsize=13)

for ax, perm, color, label in [
    (axes[0], perm_tng_holdout, "#4C72B0", "TNG300 (holdout)"),
    (axes[1], perm_flam_test,   "#E07B3F", "FLAMINGO (test)"),
]:
    means = perm.importances_mean[sort_idx]
    stds  = perm.importances_std[sort_idx]
    feats = [FEATURES[i] for i in sort_idx]
    ax.barh(feats[::-1], means[::-1], xerr=stds[::-1],
            color=color, alpha=0.8, capsize=4)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Mean R² decrease", fontsize=11)
    ax.set_title(label, fontsize=11)
    ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_12_2.pdf", bbox_inches="tight")
plt.show()

# Cross-direction summary 
print("\nCross-sim transfer summary:")
print(f"  FLAMINGO→TNG:  CV R²={np.mean(cv_scores):.4f}  Transfer R²={r2_tng:.4f}  Drop={np.mean(cv_scores)-r2_tng:.4f}")
print(f"  TNG→FLAMINGO:  CV R²={np.mean(cv_scores_tng):.4f}  "
      f"Transfer R²={r2_flam:.4f}  Drop={np.mean(cv_scores_tng)-r2_flam:.4f}")
print("Compare the measured transfer R² in both directions; within-simulation CV alone does not establish transfer asymmetry.")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

Similar to before training on TNG300 underestimates the virial mass.


In [ ]:
# Exploratory row-bootstrap importance stability, retained for comparison.
# This diagnostic is not a host-independent confidence interval; use the grouped
# CV/holdout cells for model-performance assessment.
N_BOOT    = 200
MAX_TRAIN = 10_000   
MAX_VAL   = 5_000

rng = np.random.default_rng(42)
boot_importances = np.zeros((N_BOOT, len(FEATURES)))

print(f"Running {N_BOOT} bootstrap resamples (subsampled)...")
for b in range(N_BOOT):
    # subsample then bootstrap
    sub_idx = rng.choice(len(X_flam), size=MAX_TRAIN, replace=False)
    X_sub, y_sub = X_flam[sub_idx], y_flam[sub_idx]
    
    boot_idx = rng.choice(MAX_TRAIN, size=MAX_TRAIN, replace=True)
    X_b, y_b = X_sub[boot_idx], y_sub[boot_idx]
    
    oob_mask = np.ones(MAX_TRAIN, dtype=bool)
    oob_mask[np.unique(boot_idx)] = False
    oob_idx = np.where(oob_mask)[0][:MAX_VAL]
    
    rf_b = RandomForestRegressor(**RF_PARAMS)
    rf_b.fit(X_b, y_b)
    
    perm_b = permutation_importance(
        rf_b, X_sub[oob_idx], y_sub[oob_idx],
        n_repeats=5, random_state=b, n_jobs=1
    )
    boot_importances[b] = perm_b.importances_mean
    
    if (b + 1) % 20 == 0:
        print(f"  {b+1}/{N_BOOT}")
print("Done.")

median_imp  = np.median(boot_importances, axis=0)
sort_idx    = np.argsort(median_imp)[::-1]
sorted_feat = [FEATURES[i] for i in sort_idx]
sorted_boot = boot_importances[:, sort_idx]

fig, ax = plt.subplots(figsize=(12, 7))
bp = ax.boxplot(
    sorted_boot,
    vert=False,
    patch_artist=True,
    labels=sorted_feat,
    medianprops=dict(color="black", lw=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
)
for patch in bp["boxes"]:
    patch.set_facecolor("#E07B3F")
    patch.set_alpha(0.7)

ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Permutation Importance (Mean R² decrease)", fontsize=11)
ax.set_title(f"Bootstrap Stability — FLAMINGO (N={N_BOOT} resamples)", fontsize=12)
ax.tick_params(labelsize=10)
plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_14.pdf", bbox_inches="tight")
plt.show()

print(f"\n{'Feature':<16} {'Median PI':>10} {'IQR':>10}  {'Stable?'}")
for i in sort_idx:
    med = np.median(boot_importances[:, i])
    iqr = np.percentile(boot_importances[:, i], 75) - \
          np.percentile(boot_importances[:, i], 25)
    stable = "✓" if (med > 0.005 and iqr / (med + 1e-9) < 0.5) else "unstable"
    print(f"{FEATURES[i]:<16} {med:>10.4f} {iqr:>10.4f}  {stable}")

In [ ]:
# Kinematic plane — vr vs vt
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Kinematic Plane — Pair-level", fontsize=13)

for ax, df, color, label in [
    (axes[0], flam_df_clean, "#E07B3F", f"FLAMINGO (N={len(flam_df_clean):,})"),
    (axes[1], tng_df_clean,  "#4C72B0", f"TNG300-1 (N={len(tng_df_clean):,})"),
]:
    hb = ax.hexbin(df["vt"], df["vr"], gridsize=40,
                   cmap="Greys", mincnt=1,
                   norm=mcolors.LogNorm(), linewidths=0.2)
    # selection box
    ax.axhline(-400, color=color, ls="--", lw=1, alpha=0.7)
    ax.axhline(0,    color=color, ls="--", lw=1, alpha=0.7)
    ax.axvline(0,    color=color, ls="--", lw=1, alpha=0.7)
    ax.axvline(300,  color=color, ls="--", lw=1, alpha=0.7)
    ax.set_xlabel("$v_t$ [km/s]", fontsize=11)
    ax.set_ylabel("$v_r$ [km/s]", fontsize=11)
    ax.set_title(label, fontsize=11)
    plt.colorbar(hb, ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_15.pdf", bbox_inches="tight")
plt.show()

In [ ]:
#  Separation vs kinematics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Separation vs Kinematics — Pair-level", fontsize=13)

for col, (df, color, label) in enumerate([
    (flam_df_clean, "#E07B3F", "FLAMINGO"),
    (tng_df_clean,  "#4C72B0", "TNG300-1"),
]):
    for row, (ykey, ylabel) in enumerate([
        ("vr", "$v_r$ [km/s]"),
        ("vt", "$v_t$ [km/s]"),
    ]):
        ax = axes[row, col]
        hb = ax.hexbin(df["separation"], df[ykey], gridsize=35,
                       cmap="Greys", mincnt=1,
                       norm=mcolors.LogNorm(), linewidths=0.2)
        r, _ = spearmanr(df["separation"], df[ykey])
        ax.set_xlabel("Separation [kpc]", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f"{label}  Spearman r={r:.3f}", fontsize=10)
        plt.colorbar(hb, ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_16.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Population tracers g-r vs metallicity
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Population Tracers: g-r vs Metallicity (pair mean)", fontsize=13)

for ax, df, color, label in [
    (axes[0], flam_df_clean, "#E07B3F", f"FLAMINGO (N={len(flam_df_clean):,})"),
    (axes[1], tng_df_clean,  "#4C72B0", f"TNG300-1 (N={len(tng_df_clean):,})"),
]:
    hb = ax.hexbin(df["met_mean"], df["gr_mean"], gridsize=35,
                   cmap="Greys", mincnt=1,
                   norm=mcolors.LogNorm(), linewidths=0.2)
    r, _ = spearmanr(df["met_mean"].dropna(), df["gr_mean"].dropna())
    ax.axhline(0.68, color="gray", ls="--", lw=1, alpha=0.7, label="g-r = 0.68")
    ax.set_xlabel("Pair mean log10(Z / 0.0127) [dex]", fontsize=11)
    ax.set_ylabel("g - r (pair mean)", fontsize=11)
    ax.set_title(f"{label}  Spearman r={r:.3f}", fontsize=11)
    ax.legend(fontsize=9)
    plt.colorbar(hb, ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_17.pdf", bbox_inches="tight")
plt.show()

In [ ]:
#  Morphology vs kinematics: sigma_mean vs kappa_mean 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(r"Morphology vs Kinematics: $\sigma_\star$ vs $\kappa_{\rm co}$ (pair mean)", fontsize=13)

for ax, df, color, label in [
    (axes[0], flam_df_clean, "#E07B3F", f"FLAMINGO (N={len(flam_df_clean):,})"),
    (axes[1], tng_df_clean,  "#4C72B0", f"TNG300-1 (N={len(tng_df_clean):,})"),
]:
    valid = df["sigma_mean"].notna() & df["kappa_mean"].notna()
    hb = ax.hexbin(df.loc[valid, "kappa_mean"], df.loc[valid, "sigma_mean"],
                   gridsize=35, cmap="Greys", mincnt=1,
                   norm=mcolors.LogNorm(), linewidths=0.2)
    r, _ = spearmanr(df.loc[valid, "kappa_mean"], df.loc[valid, "sigma_mean"])
    ax.axvline(0.35, color="gray", ls="--", lw=1, alpha=0.7,
               label=r"$\kappa_{\rm co}=0.35$")
    ax.set_xlabel(r"$\kappa_{\rm co}$ (pair mean)", fontsize=11)
    ax.set_ylabel(r"$\sigma_\star$ [km/s] (pair mean)", fontsize=11)
    ax.set_title(f"{label}  Spearman r={r:.3f}", fontsize=11)
    ax.legend(fontsize=9)
    plt.colorbar(hb, ax=ax, label="Counts (log)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_18.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Target log M200c vs each feature 
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle(r"$\log M_{200c}$ vs Each Feature", fontsize=14)
axes = axes.flatten()

for k, feat in enumerate(FEATURES):
    ax = axes[k]
    for df, color, label in [
        (flam_df_clean, "#E07B3F", "FLAMINGO"),
        (tng_df_clean,  "#4C72B0", "TNG300-1"),
    ]:
        valid = df[feat].notna() & df[TARGET].notna()
        r, _ = spearmanr(df.loc[valid, feat], df.loc[valid, TARGET])
        ax.scatter(df.loc[valid, feat], df.loc[valid, TARGET],
                   alpha=0.15, s=3, color=color,
                   label=f"{label} r={r:.2f}")
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel(r"$\log M_{200c}$", fontsize=9)
    ax.legend(fontsize=7, markerscale=3)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_19.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Spearman correlation matrix 
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Spearman Correlation Matrix", fontsize=13)

all_cols = FEATURES + [TARGET]

for ax, df, label in [
    (axes[0], flam_df_clean, "FLAMINGO"),
    (axes[1], tng_df_clean,  "TNG300-1"),
]:
    corr = df[all_cols].apply(
        lambda col: df[all_cols].apply(
            lambda other: spearmanr(
                col.dropna().align(other.dropna(), join="inner")[0],
                col.dropna().align(other.dropna(), join="inner")[1]
            )[0]
        )
    )
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, mask=mask,
                annot=True, fmt=".2f", cmap="RdBu_r",
                vmin=-1, vmax=1, center=0,
                linewidths=0.5, annot_kws={"size": 7},
                cbar_kws={"shrink": 0.8})
    ax.set_title(label, fontsize=12)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_20.pdf", bbox_inches="tight")
plt.show()

In [ ]:
#  Feature block ablation study 

BLOCKS = {
    "core_dynamics": [
        "separation", "vr", "vt", "log_mass_ratio"
    ],
    "core_chemistry": [
        "separation", "vr", "vt", "log_mass_ratio",
        "met_mean", "met_diff"
    ],
    "core_morphology": [
        "separation", "vr", "vt", "log_mass_ratio",
        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"
    ],
    "core_photometry": [
        "separation", "vr", "vt", "log_mass_ratio",
        "gr_mean", "gr_diff"
    ],
}

kf = GroupKFold(n_splits=5)
ablation_results = []

for block_name, feats in BLOCKS.items():
    print(f"\n {block_name} {feats} ")

    X_f = flam_df_clean[feats].values
    y_f = flam_df_clean[TARGET].values
    X_t = tng_df_clean[feats].dropna().index
    X_t = tng_df_clean.loc[tng_df_clean[feats].notna().all(axis=1), feats].values
    y_t = tng_df_clean.loc[tng_df_clean[feats].notna().all(axis=1), TARGET].values

    # FLAMINGO 5-fold CV
    cv_r2 = []
    for train_idx, val_idx in kf.split(X_f, groups=flam_groups):
        rf = RandomForestRegressor(**RF_PARAMS)
        rf.fit(X_f[train_idx], y_f[train_idx])
        cv_r2.append(r2_score(y_f[val_idx], rf.predict(X_f[val_idx])))
    flam_cv = np.mean(cv_r2)
    flam_cv_std = np.std(cv_r2)
    print(f"  FLAMINGO CV R²: {flam_cv:.4f} ± {flam_cv_std:.4f}")

    # fit on full FLAMINGO, test on TNG
    rf_full = RandomForestRegressor(**RF_PARAMS)
    rf_full.fit(X_f, y_f)
    tng_r2 = r2_score(y_t, rf_full.predict(X_t))
    drop   = flam_cv - tng_r2
    print(f"  TNG test R²:    {tng_r2:.4f}")
    print(f"  Cross-sim drop: {drop:.4f}")

    # permutation importance on TNG 
    perm = permutation_importance(
        rf_full, X_t, y_t,
        n_repeats=20, random_state=42, n_jobs=1
    )
    sort_idx = perm.importances_mean.argsort()[::-1]
    print(f"  Feature importances (TNG test):")
    for i in sort_idx:
        print(f"    {feats[i]:<16} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

    ablation_results.append({
        "block":       block_name,
        "n_features":  len(feats),
        "flam_cv_r2":  flam_cv,
        "flam_cv_std": flam_cv_std,
        "tng_r2":      tng_r2,
        "drop":        drop,
        "features":    feats,
        "perm_means":  perm.importances_mean[sort_idx],
        "perm_feats":  [feats[i] for i in sort_idx],
    })

# summary table
print("\n\nAblation Summary")
print(f"{'Block':<20} {'N feats':>8} {'FLAM CV R²':>12} {'TNG R²':>10} {'Drop':>8}  {'Verdict'}")

for r in sorted(ablation_results, key=lambda x: x["drop"]):
    verdict = ("✓ best transfer" if r["drop"] == min(x["drop"] for x in ablation_results)
               else "")
    print(f"{r['block']:<20} {r['n_features']:>8} "
          f"{r['flam_cv_r2']:>12.4f} {r['tng_r2']:>10.4f} "
          f"{r['drop']:>8.4f}  {verdict}")

# visual summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ablation Study: Feature Blocks", fontsize=13)

block_names = [r["block"] for r in ablation_results]
flam_cvs    = [r["flam_cv_r2"] for r in ablation_results]
tng_r2s     = [r["tng_r2"]     for r in ablation_results]
drops       = [r["drop"]        for r in ablation_results]

x = np.arange(len(BLOCKS))
w = 0.35

axes[0].bar(x - w/2, flam_cvs, w, label="FLAMINGO CV R²",
            color="#E07B3F", alpha=0.8)
axes[0].bar(x + w/2, tng_r2s,  w, label="TNG test R²",
            color="#4C72B0", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("R²", fontsize=11)
axes[0].set_title("R² by feature block", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

axes[1].bar(x, drops, color="steelblue", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Cross-sim R² drop", fontsize=11)
axes[1].set_title("Transfer stability (lower = better)", fontsize=11)
# mark smallest drop
best_idx = np.argmin(drops)
axes[1].bar(best_idx, drops[best_idx], color="#E07B3F",
            alpha=0.9, label="best transfer")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_21.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Ablation study train TNG, test FLAMINGO

ablation_results_rev = []

for block_name, feats in BLOCKS.items():
    print(f"\n {block_name} ")

    # TNG train
    tng_valid = tng_df_clean[feats].notna().all(axis=1)
    X_t = tng_df_clean.loc[tng_valid, feats].values
    y_t = tng_df_clean.loc[tng_valid, TARGET].values

    # FLAMINGO test
    flam_valid = flam_df_clean[feats].notna().all(axis=1)
    X_f = flam_df_clean.loc[flam_valid, feats].values
    y_f = flam_df_clean.loc[flam_valid, TARGET].values

    # TNG 5-fold CV
    cv_r2 = []
    for train_idx, val_idx in kf.split(X_t, groups=tng_groups[tng_valid.to_numpy()]):
        rf = RandomForestRegressor(**RF_PARAMS)
        rf.fit(X_t[train_idx], y_t[train_idx])
        cv_r2.append(r2_score(y_t[val_idx], rf.predict(X_t[val_idx])))
    tng_cv     = np.mean(cv_r2)
    tng_cv_std = np.std(cv_r2)
    print(f"  TNG CV R²:       {tng_cv:.4f} ± {tng_cv_std:.4f}")

    # fit on full TNG, test on FLAMINGO
    rf_full   = RandomForestRegressor(**RF_PARAMS)
    rf_full.fit(X_t, y_t)
    flam_r2   = r2_score(y_f, rf_full.predict(X_f))
    drop      = tng_cv - flam_r2
    print(f"  FLAMINGO test R²: {flam_r2:.4f}")
    print(f"  Cross-sim drop:   {drop:.4f}")

    # permutation importance on FLAMINGO
    perm = permutation_importance(
        rf_full, X_f, y_f,
        n_repeats=20, random_state=42, n_jobs=1
    )
    sort_idx = perm.importances_mean.argsort()[::-1]
    print(f"  Feature importances (FLAMINGO test):")
    for i in sort_idx:
        print(f"    {feats[i]:<16} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

    ablation_results_rev.append({
        "block":       block_name,
        "n_features":  len(feats),
        "tng_cv_r2":   tng_cv,
        "tng_cv_std":  tng_cv_std,
        "flam_r2":     flam_r2,
        "drop":        drop,
        "features":    feats,
        "perm_means":  perm.importances_mean[sort_idx],
        "perm_feats":  [feats[i] for i in sort_idx],
    })

# summary table 
print("\n\n Reverse Ablation Summary (train TNG → test FLAMINGO) ")
print(f"{'Block':<20} {'N feats':>8} {'TNG CV R²':>12} {'FLAM R²':>10} {'Drop':>8}  {'Verdict'}")

for r in sorted(ablation_results_rev, key=lambda x: x["drop"]):
    verdict = ("✓ best transfer" if r["drop"] == min(x["drop"] for x in ablation_results_rev)
               else "")
    print(f"{r['block']:<20} {r['n_features']:>8} "
          f"{r['tng_cv_r2']:>12.4f} {r['flam_r2']:>10.4f} "
          f"{r['drop']:>8.4f}  {verdict}")

# visual summary 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Reverse Ablation Study: Train TNG → Test FLAMINGO", fontsize=13)

block_names = [r["block"] for r in ablation_results_rev]
tng_cvs     = [r["tng_cv_r2"] for r in ablation_results_rev]
flam_r2s    = [r["flam_r2"]   for r in ablation_results_rev]
drops_rev   = [r["drop"]      for r in ablation_results_rev]

x = np.arange(len(BLOCKS))
w = 0.35

axes[0].bar(x - w/2, tng_cvs,  w, label="TNG CV R²",
            color="#4C72B0", alpha=0.8)
axes[0].bar(x + w/2, flam_r2s, w, label="FLAMINGO test R²",
            color="#E07B3F", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("R²", fontsize=11)
axes[0].set_title("R² by feature block", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

axes[1].bar(x, drops_rev, color="steelblue", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Cross-sim R² drop", fontsize=11)
axes[1].set_title("Transfer stability (lower = better)", fontsize=11)
best_idx = np.argmin(drops_rev)
axes[1].bar(best_idx, drops_rev[best_idx], color="#E07B3F",
            alpha=0.9, label="best transfer")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_22.pdf", bbox_inches="tight")
plt.show()

# combined direction comparison
print("\n Both directions combined")
print(f"{'Block':<20} {'FLAM→TNG drop':>15} {'TNG→FLAM drop':>15} {'Max drop':>10}  {'Verdict'}")


combined = []
for r_fwd, r_rev in zip(
    sorted(ablation_results,     key=lambda x: x["block"]),
    sorted(ablation_results_rev, key=lambda x: x["block"])
):
    max_drop = max(r_fwd["drop"], r_rev["drop"])
    combined.append((r_fwd["block"], r_fwd["drop"], r_rev["drop"], max_drop))

for block, drop_fwd, drop_rev, max_drop in sorted(combined, key=lambda x: x[3]):
    verdict = "✓ most robust" if max_drop == min(x[3] for x in combined) else ""
    print(f"{block:<20} {drop_fwd:>15.4f} {drop_rev:>15.4f} {max_drop:>10.4f}  {verdict}")

best_combined = min(combined, key=lambda x: x[3])
print(f"\nMost robust block overall: {best_combined[0]}")
print(f"  Max cross-sim drop in either direction: {best_combined[3]:.4f}")

In [ ]:
# Feature block ablation study

BLOCKS = {
    "core_dynamics": [
        "separation", "vr", "vt"
    ],
    "core_chemistry": [
        "separation", "vr", "vt", 
        "met_mean", "met_diff"
    ],
    "core_morphology": [
        "separation", "vr", "vt", 
        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"
    ],
    "core_photometry": [
        "separation", "vr", "vt", 
        "gr_mean", "gr_diff"
    ],
}

kf = GroupKFold(n_splits=5)
ablation_results = []

for block_name, feats in BLOCKS.items():
    print(f"\n {block_name} {feats} ")

    X_f = flam_df_clean[feats].values
    y_f = flam_df_clean[TARGET].values
    X_t = tng_df_clean[feats].dropna().index
    X_t = tng_df_clean.loc[tng_df_clean[feats].notna().all(axis=1), feats].values
    y_t = tng_df_clean.loc[tng_df_clean[feats].notna().all(axis=1), TARGET].values

    # FLAMINGO 5-fold CV
    cv_r2 = []
    for train_idx, val_idx in kf.split(X_f, groups=flam_groups):
        rf = RandomForestRegressor(**RF_PARAMS)
        rf.fit(X_f[train_idx], y_f[train_idx])
        cv_r2.append(r2_score(y_f[val_idx], rf.predict(X_f[val_idx])))
    flam_cv = np.mean(cv_r2)
    flam_cv_std = np.std(cv_r2)
    print(f"  FLAMINGO CV R²: {flam_cv:.4f} ± {flam_cv_std:.4f}")

    # fit on full FLAMINGO, test on TNG
    rf_full = RandomForestRegressor(**RF_PARAMS)
    rf_full.fit(X_f, y_f)
    tng_r2 = r2_score(y_t, rf_full.predict(X_t))
    drop   = flam_cv - tng_r2
    print(f"  TNG test R²:    {tng_r2:.4f}")
    print(f"  Cross-sim drop: {drop:.4f}")

    # permutation importance on TNG 
    perm = permutation_importance(
        rf_full, X_t, y_t,
        n_repeats=20, random_state=42, n_jobs=1
    )
    sort_idx = perm.importances_mean.argsort()[::-1]
    print(f"  Feature importances (TNG test):")
    for i in sort_idx:
        print(f"    {feats[i]:<16} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

    ablation_results.append({
        "block":       block_name,
        "n_features":  len(feats),
        "flam_cv_r2":  flam_cv,
        "flam_cv_std": flam_cv_std,
        "tng_r2":      tng_r2,
        "drop":        drop,
        "features":    feats,
        "perm_means":  perm.importances_mean[sort_idx],
        "perm_feats":  [feats[i] for i in sort_idx],
    })

# summary table
print("\n\n Ablation Summary")
print(f"{'Block':<20} {'N feats':>8} {'FLAM CV R²':>12} {'TNG R²':>10} {'Drop':>8}  {'Verdict'}")

for r in sorted(ablation_results, key=lambda x: x["drop"]):
    verdict = ("✓ best transfer" if r["drop"] == min(x["drop"] for x in ablation_results)
               else "")
    print(f"{r['block']:<20} {r['n_features']:>8} "
          f"{r['flam_cv_r2']:>12.4f} {r['tng_r2']:>10.4f} "
          f"{r['drop']:>8.4f}  {verdict}")

# visual summary 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ablation Study: Feature Blocks", fontsize=13)

block_names = [r["block"] for r in ablation_results]
flam_cvs    = [r["flam_cv_r2"] for r in ablation_results]
tng_r2s     = [r["tng_r2"]     for r in ablation_results]
drops       = [r["drop"]        for r in ablation_results]

x = np.arange(len(BLOCKS))
w = 0.35

axes[0].bar(x - w/2, flam_cvs, w, label="FLAMINGO CV R²",
            color="#E07B3F", alpha=0.8)
axes[0].bar(x + w/2, tng_r2s,  w, label="TNG test R²",
            color="#4C72B0", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("R²", fontsize=11)
axes[0].set_title("R² by feature block", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

axes[1].bar(x, drops, color="steelblue", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Cross-sim R² drop", fontsize=11)
axes[1].set_title("Transfer stability (lower = better)", fontsize=11)
# mark smallest drop
best_idx = np.argmin(drops)
axes[1].bar(best_idx, drops[best_idx], color="#E07B3F",
            alpha=0.9, label="best transfer")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_23.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Ablation study train TNG, test FLAMINGO

ablation_results_rev = []

for block_name, feats in BLOCKS.items():
    print(f"\n── {block_name} ──")

    # TNG train
    tng_valid = tng_df_clean[feats].notna().all(axis=1)
    X_t = tng_df_clean.loc[tng_valid, feats].values
    y_t = tng_df_clean.loc[tng_valid, TARGET].values

    # FLAMINGO test
    flam_valid = flam_df_clean[feats].notna().all(axis=1)
    X_f = flam_df_clean.loc[flam_valid, feats].values
    y_f = flam_df_clean.loc[flam_valid, TARGET].values

    # TNG 5-fold CV
    cv_r2 = []
    for train_idx, val_idx in kf.split(X_t, groups=tng_groups[tng_valid.to_numpy()]):
        rf = RandomForestRegressor(**RF_PARAMS)
        rf.fit(X_t[train_idx], y_t[train_idx])
        cv_r2.append(r2_score(y_t[val_idx], rf.predict(X_t[val_idx])))
    tng_cv     = np.mean(cv_r2)
    tng_cv_std = np.std(cv_r2)
    print(f"  TNG CV R²:       {tng_cv:.4f} ± {tng_cv_std:.4f}")

    # fit on full TNG, test on FLAMINGO
    rf_full   = RandomForestRegressor(**RF_PARAMS)
    rf_full.fit(X_t, y_t)
    flam_r2   = r2_score(y_f, rf_full.predict(X_f))
    drop      = tng_cv - flam_r2
    print(f"  FLAMINGO test R²: {flam_r2:.4f}")
    print(f"  Cross-sim drop:   {drop:.4f}")

    # permutation importance on FLAMINGO 
    perm = permutation_importance(
        rf_full, X_f, y_f,
        n_repeats=20, random_state=42, n_jobs=1
    )
    sort_idx = perm.importances_mean.argsort()[::-1]
    print(f"  Feature importances (FLAMINGO test):")
    for i in sort_idx:
        print(f"    {feats[i]:<16} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

    ablation_results_rev.append({
        "block":       block_name,
        "n_features":  len(feats),
        "tng_cv_r2":   tng_cv,
        "tng_cv_std":  tng_cv_std,
        "flam_r2":     flam_r2,
        "drop":        drop,
        "features":    feats,
        "perm_means":  perm.importances_mean[sort_idx],
        "perm_feats":  [feats[i] for i in sort_idx],
    })

# summary table
print("\n\n── Reverse Ablation Summary (train TNG → test FLAMINGO) ──")
print(f"{'Block':<20} {'N feats':>8} {'TNG CV R²':>12} {'FLAM R²':>10} {'Drop':>8}  {'Verdict'}")


for r in sorted(ablation_results_rev, key=lambda x: x["drop"]):
    verdict = ("✓ best transfer" if r["drop"] == min(x["drop"] for x in ablation_results_rev)
               else "")
    print(f"{r['block']:<20} {r['n_features']:>8} "
          f"{r['tng_cv_r2']:>12.4f} {r['flam_r2']:>10.4f} "
          f"{r['drop']:>8.4f}  {verdict}")

# visual summary 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Reverse Ablation Study: Train TNG → Test FLAMINGO", fontsize=13)

block_names = [r["block"] for r in ablation_results_rev]
tng_cvs     = [r["tng_cv_r2"] for r in ablation_results_rev]
flam_r2s    = [r["flam_r2"]   for r in ablation_results_rev]
drops_rev   = [r["drop"]      for r in ablation_results_rev]

x = np.arange(len(BLOCKS))
w = 0.35

axes[0].bar(x - w/2, tng_cvs,  w, label="TNG CV R²",
            color="#4C72B0", alpha=0.8)
axes[0].bar(x + w/2, flam_r2s, w, label="FLAMINGO test R²",
            color="#E07B3F", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("R²", fontsize=11)
axes[0].set_title("R² by feature block", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

axes[1].bar(x, drops_rev, color="steelblue", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(block_names, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Cross-sim R² drop", fontsize=11)
axes[1].set_title("Transfer stability (lower = better)", fontsize=11)
best_idx = np.argmin(drops_rev)
axes[1].bar(best_idx, drops_rev[best_idx], color="#E07B3F",
            alpha=0.9, label="best transfer")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_24.pdf", bbox_inches="tight")
plt.show()

# combined direction comparison 
print("\n Both directions combined")
print(f"{'Block':<20} {'FLAM→TNG drop':>15} {'TNG→FLAM drop':>15} {'Max drop':>10}  {'Verdict'}")
print("-" * 70)

combined = []
for r_fwd, r_rev in zip(
    sorted(ablation_results,     key=lambda x: x["block"]),
    sorted(ablation_results_rev, key=lambda x: x["block"])
):
    max_drop = max(r_fwd["drop"], r_rev["drop"])
    combined.append((r_fwd["block"], r_fwd["drop"], r_rev["drop"], max_drop))

for block, drop_fwd, drop_rev, max_drop in sorted(combined, key=lambda x: x[3]):
    verdict = "✓ most robust" if max_drop == min(x[3] for x in combined) else ""
    print(f"{block:<20} {drop_fwd:>15.4f} {drop_rev:>15.4f} {max_drop:>10.4f}  {verdict}")

best_combined = min(combined, key=lambda x: x[3])
print(f"\nMost robust block overall: {best_combined[0]}")
print(f"  Max cross-sim drop in either direction: {best_combined[3]:.4f}")

In [ ]:
print(flam_df_clean[["kappa_mean", "sigma_mean", "kappa_diff", "sigma_diff"]].corr())